# Lượt 4 — Frozen learned depth head trên Official CDCN

**Câu hỏi:** đọc cùng predicted depth map bằng head 1.265 tham số có tốt hơn lấy mean không?

E1: `CASIA_E1_CDCN_OFFICIAL_seed42_20260921T114029Z`. E2 nạp checkpoint E1,
khóa cả weights và BatchNorm, train riêng head với BCE 10 epoch. Mọi lựa chọn
checkpoint và threshold dùng validation. Notebook không train lại E1 hoặc sinh lại
3DDFA. CASIA test đã được xem ở Lượt 3: bảng mới là **so sánh phát triển**, chưa phải
xác nhận độc lập, nhiều seed hay SOTA. UCDCN 2024 là tiền lệ của learned depth classifier.

Chọn GPU runtime. Mở notebook này và chạy tuần tự. Chuẩn bị file
`DeepFace-PAD-Round4-source.zip` nếu runtime chưa có source mới.
Không dùng kết quả test để chọn phiên bản, epoch, threshold hay sửa config.


In [ ]:
#@title 0.1 Drive, helper và đường dẫn
from google.colab import drive, files
drive.mount("/content/drive")
import importlib, json, shutil, subprocess, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

REPO_ROOT = Path("/content/round4-source/DeepFace-PAD")
DRIVE_ROOT = Path("/content/drive/MyDrive/face-pad")
RUNS_DIR = DRIVE_ROOT / "runs"
REPORTS_DIR = DRIVE_ROOT / "reports"
DATA_ROOT = Path("/content/datasets/casia-fasd")
DEPTH_BACKUP = DRIVE_ROOT / "depth-checkpoints/casia-fasd"
DEPTH_ROOT = DATA_ROOT / "depth"
DEPTH_LEDGER = DEPTH_ROOT / "depth_status.csv"
DEPTH_QA = REPORTS_DIR / "casia-depth-qa-round4.json"
MANIFEST = REPO_ROOT / "data/manifests/casia_fasd_debug.csv"
DEPTH_MANIFEST = REPO_ROOT / "data/manifests/casia_fasd_debug_with_depth.csv"
E1_RUN = RUNS_DIR / "CASIA_E1_CDCN_OFFICIAL_seed42_20260921T114029Z"
for directory in (RUNS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

def run(command, cwd=None):
    subprocess.run([str(item) for item in command], cwd=str(cwd) if cwd else None, check=True)


In [ ]:
#@title 0.2 Nạp source đã commit từ ZIP và chạy tests
archive = Path("/content/DeepFace-PAD-Round4-source.zip")
if not archive.is_file():
    print("Upload DeepFace-PAD-Round4-source.zip vừa được cung cấp cùng notebook.")
    uploaded = files.upload()
    assert archive.is_file(), "Thiếu đúng file DeepFace-PAD-Round4-source.zip"
shutil.unpack_archive(str(archive), "/content/round4-source")
assert (REPO_ROOT / "src/deepface_pad/models/cdcn_depth_head.py").is_file()
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], cwd=REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))
importlib.invalidate_caches()
run([sys.executable, "-m", "pytest", "-q"], cwd=REPO_ROOT)
import torch
assert torch.cuda.is_available(), "Chọn Runtime > Change runtime type > GPU"
print("GPU:", torch.cuda.get_device_name(0))


## 1. Khôi phục cùng dataset và pseudo-depth E1
Các cell này được sao chép từ notebook E1 đã commit; output notebook E1 đang mở không bị thay đổi.


In [ ]:
#@title 1.1 Chuẩn bị CASIA-FASD ở đường dẫn chuẩn
def looks_like_casia(path):
    return (path / "train" / "live").is_dir() and (path / "test" / "spoof").is_dir()

def find_casia_root(base):
    if looks_like_casia(base):
        return base
    if base.exists():
        for train_dir in base.rglob("train"):
            if looks_like_casia(train_dir.parent):
                return train_dir.parent
    return None

if not looks_like_casia(DATA_ROOT):
    drive_dataset = DRIVE_ROOT / "datasets" / "casia-fasd"
    dataset_zip = DRIVE_ROOT / "casia-fasd.zip"
    DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
    if looks_like_casia(drive_dataset):
        shutil.copytree(drive_dataset, DATA_ROOT, dirs_exist_ok=True)
    elif dataset_zip.is_file():
        shutil.unpack_archive(str(dataset_zip), str(DATA_ROOT.parent))
        discovered = find_casia_root(DATA_ROOT.parent)
        assert discovered is not None, "Không tìm thấy CASIA root sau khi giải nén"
        if discovered.resolve() != DATA_ROOT.resolve():
            shutil.copytree(discovered, DATA_ROOT, dirs_exist_ok=True)
    else:
        run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
        import kagglehub
        downloaded = Path(kagglehub.dataset_download("immada/casia-fasd"))
        roots = [p.parent for p in downloaded.rglob("train") if looks_like_casia(p.parent)]
        assert roots, "Không tìm thấy CASIA root trong Kaggle download"
        shutil.copytree(roots[0], DATA_ROOT, dirs_exist_ok=True)

assert looks_like_casia(DATA_ROOT), f"CASIA chưa đúng cấu trúc: {DATA_ROOT}"
print("Dataset:", DATA_ROOT)


In [ ]:
#@title 1.2 Tạo lại manifest xác định và kiểm tra protocol
run([
    sys.executable, "scripts/prepare_casia_fasd.py",
    "--data-root", DATA_ROOT,
    "--output-dir", REPO_ROOT / "data/manifests",
    "--frames-per-video", "20",
    "--val-subjects", "4,9,14,19",
], cwd=REPO_ROOT)
run([sys.executable, "scripts/validate_manifest.py", MANIFEST, "--data-root", DATA_ROOT, "--subject-disjoint"], cwd=REPO_ROOT)


In [ ]:
#@title 1.3 Khôi phục cache depth từ Drive; không tái sinh map hợp lệ
assert DEPTH_BACKUP.is_dir(), f"Thiếu depth checkpoint trên Drive: {DEPTH_BACKUP}"
DEPTH_ROOT.mkdir(parents=True, exist_ok=True)

# Google Drive có độ trễ cao với hàng nghìn file nhỏ. Copy từng file, bỏ qua file
# đã đủ byte và in tiến độ để cell có thể resume sau khi runtime bị ngắt.
backup_files = [path for path in DEPTH_BACKUP.iterdir() if path.is_file()]
copied = skipped = 0
for index, source in enumerate(backup_files, start=1):
    destination = DEPTH_ROOT / source.name
    if destination.is_file() and destination.stat().st_size == source.stat().st_size:
        skipped += 1
    else:
        shutil.copy2(source, destination)
        copied += 1
    if index % 250 == 0 or index == len(backup_files):
        print(f"Depth restore {index}/{len(backup_files)} — copied={copied}, reused={skipped}", flush=True)

# Checkpoint Drive chỉ giữ 3.000 live maps để tránh 9.000 file zero dư thừa.
# Lệnh này tạo lại attack zero-maps ở local, tái sử dụng toàn bộ live maps hợp lệ
# và làm mới ledger; nó không chạy 3DDFA.
run([
    sys.executable, "scripts/generate_depth.py", MANIFEST,
    "--data-root", DATA_ROOT,
    "--output-root", DEPTH_ROOT,
], cwd=REPO_ROOT)

assert DEPTH_LEDGER.is_file(), f"Thiếu ledger: {DEPTH_LEDGER}"

ledger = pd.read_csv(DEPTH_LEDGER, keep_default_na=False, dtype={"sample_id": str})
display(ledger.groupby("status").size().rename("samples").to_frame())
pending = int((ledger.status == "pending_3ddfa").sum())
failed = int((ledger.status == "failed_bona_fide").sum())
complete_live = int((ledger.status == "complete_bona_fide").sum())
assert pending == 0 and failed == 0 and complete_live == 3000, (pending, failed, complete_live)


In [ ]:
#@title 1.4 Materialize manifest, provenance và full depth QA
run([
    sys.executable, "scripts/materialize_depth_manifest.py", MANIFEST,
    "--ledger", DEPTH_LEDGER,
    "--data-root", DATA_ROOT,
    "--output", DEPTH_MANIFEST,
], cwd=REPO_ROOT)
run([
    sys.executable, "scripts/verify_depth_provenance.py",
    "--source", MANIFEST,
    "--ledger", DEPTH_LEDGER,
    "--derived", DEPTH_MANIFEST,
], cwd=REPO_ROOT)
run([
    sys.executable, "scripts/validate_manifest.py", DEPTH_MANIFEST,
    "--data-root", DATA_ROOT, "--require-depth",
], cwd=REPO_ROOT)
run([
    sys.executable, "scripts/audit_depth.py", DEPTH_MANIFEST,
    "--data-root", DATA_ROOT, "--report", DEPTH_QA,
], cwd=REPO_ROOT)
print("Depth QA:", DEPTH_QA)


## 2. Xác nhận đối chứng và đăng ký E2 trước khi train


In [ ]:
#@title 2.1 Kiểm tra đúng E1, manifest và preprocessing
from deepface_pad.data import manifest_checksum
required = ["best.ckpt", "config.yaml", "model_provenance.json", "manifest_checksum.json", "val_frame_scores.csv", "val_scores.csv", "threshold.json"]
assert all((E1_RUN / name).is_file() for name in required), "Thiếu frozen E1 artifact trên Drive"
e1_config = yaml.safe_load((E1_RUN / "config.yaml").read_text())
assert e1_config["model"]["name"] == "cdcn_official"
assert e1_config["data"]["normalization"] == "cdcn_official"
assert e1_config["data"]["image_size"] == 256
assert e1_config["evaluation"]["aggregation"] == "mean"
expected_manifest = json.loads((E1_RUN / "manifest_checksum.json").read_text())["sha256"]
assert manifest_checksum(DEPTH_MANIFEST) == expected_manifest, "Manifest khác E1; dừng để kiểm tra, không đổi split"
e1_state = torch.load(E1_RUN / "best.ckpt", map_location="cpu", weights_only=False)
assert e1_state["config"]["model"]["name"] == "cdcn_official"

config = yaml.safe_load((REPO_ROOT / "configs/casia_e2_official_head_frozen.yaml").read_text())
assert config["model"]["theta"] == e1_config["model"].get("theta", .7)
config["data"]["manifest"] = str(DEPTH_MANIFEST)
config["data"]["root"] = str(DATA_ROOT)
config["training"]["init_checkpoint"] = str(E1_RUN / "best.ckpt")
CONFIG_PATH = REPO_ROOT / "configs/round4_full_runtime.yaml"
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
print(yaml.safe_dump(config, sort_keys=False))
print("E1 checkpoint SHA256:", manifest_checksum(E1_RUN / "best.ckpt"))


In [ ]:
#@title 2.2 Smoke nạp checkpoint, khóa backbone và so lại validation E1
from deepface_pad.data import PadDataset
from deepface_pad.train import build_model, load_initial_weights, _set_stage, score_loader
from torch.utils.data import DataLoader
device = torch.device("cuda")
e1 = build_model(e1_config).to(device)
e1.load_state_dict(e1_state["model"])
e1.eval()
e2 = build_model(config).to(device)
load_initial_weights(e2, E1_RUN / "best.ckpt", device)
_set_stage(e2, "head")
e2.train()
assert not e2.backbone.training
assert all(torch.equal(value.cpu(), e2.backbone.state_dict()[key].cpu()) for key, value in e1_state["model"].items())
val_data = PadDataset(DEPTH_MANIFEST, DATA_ROOT, "val", 256, augment=False, normalization="cdcn_official")
val_loader = DataLoader(val_data, batch_size=8, shuffle=False, num_workers=2)
images = next(iter(val_loader))["image"].to(device)
with torch.no_grad():
    assert torch.equal(e1(images)["depth"], e2(images)["depth"])
replayed = score_loader(e1, val_loader, device)
saved = pd.read_csv(E1_RUN / "val_frame_scores.csv", dtype={"sample_id": str})
paired = saved.merge(replayed, on=["sample_id", "video_id", "label"], suffixes=("_saved", "_replayed"), validate="one_to_one")
assert len(paired) == len(saved) == len(replayed)
assert np.allclose(paired.score_saved, paired.score_replayed, atol=1e-6, rtol=1e-5), "Không tái lập được validation score E1; kiểm tra dataset/crop/version trước train"
print("PASS: same manifest, exact backbone tensors, identical depth, reproduced E1 validation")
del e1, e2, images
torch.cuda.empty_cache()


In [ ]:
#@title 2.3 Train smoke 1 epoch, rồi full 10 epoch (validation chọn best)
def registered_run(payload, path):
    path.write_text(yaml.safe_dump(payload, sort_keys=False), encoding="utf-8")
    candidates = []
    for folder in RUNS_DIR.glob(payload["experiment_id"] + "_seed42_*"):
        if all((folder / name).is_file() for name in ("best.ckpt", "metrics.json", "config.yaml", "manifest_checksum.json")):
            existing = yaml.safe_load((folder / "config.yaml").read_text())
            if existing == payload and json.loads((folder / "manifest_checksum.json").read_text())["sha256"] == expected_manifest:
                candidates.append(folder)
    if not candidates:
        run([sys.executable, "scripts/run_experiment.py", path], cwd=REPO_ROOT)
        return registered_run(payload, path)
    return sorted(candidates)[-1]

def assert_frozen_backbone(folder):
    result = torch.load(folder / "best.ckpt", map_location="cpu", weights_only=False)
    assert all(torch.equal(value, result["model"]["backbone." + key]) for key, value in e1_state["model"].items()), "Backbone hoặc BatchNorm thay đổi"
    assert json.loads((folder / "initial_checkpoint.json").read_text())["sha256"] == manifest_checksum(E1_RUN / "best.ckpt")

import copy
smoke_config = copy.deepcopy(config)
smoke_config["experiment_id"] += "_SMOKE"
smoke_config["training"]["stages"][0]["epochs"] = 1
SMOKE_RUN = registered_run(smoke_config, REPO_ROOT / "configs/round4_smoke_runtime.yaml")
assert_frozen_backbone(SMOKE_RUN)
smoke_log = pd.read_csv(SMOKE_RUN / "train_log.csv")
assert np.isfinite(smoke_log[["train_loss", "val_loss"]].to_numpy()).all()
E2_RUN = registered_run(config, CONFIG_PATH)
assert_frozen_backbone(E2_RUN)
print("Full E2:", E2_RUN)


In [ ]:
#@title 2.4 Bảng validation và BCE curve
from deepface_pad.metrics import evaluate_scores
import matplotlib.pyplot as plt
e1_val = pd.read_csv(E1_RUN / "val_scores.csv")
e1_threshold = json.loads((E1_RUN / "threshold.json").read_text())
assert e1_threshold["source"] == "validation"
e2_threshold = json.loads((E2_RUN / "threshold.json").read_text())
assert e2_threshold["source"] == "validation"
val_metrics = pd.DataFrame([
    {"model": "E1 fixed mean", **evaluate_scores(e1_val.label.to_numpy(), e1_val.score.to_numpy(), e1_threshold["threshold"]).to_dict()},
    {"model": "E2 frozen head", **json.loads((E2_RUN / "metrics.json").read_text())},
]).set_index("model")
display(val_metrics.style.format({"apcer":"{:.4%}", "bpcer":"{:.4%}", "acer":"{:.4%}", "eer":"{:.4%}", "auc":"{:.6f}", "threshold":"{:.8f}"}))
log = pd.read_csv(E2_RUN / "train_log.csv")
best = log.loc[log.val_loss.idxmin()]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(log.epoch, log.train_loss, label="train BCE")
ax.plot(log.epoch, log.val_loss, label="validation BCE")
ax.scatter([best.epoch], [best.val_loss], color="red", label="best validation epoch")
ax.set(xlabel="Epoch", ylabel="BCE loss", title="E2: head-only training")
ax.grid(alpha=.25); ax.legend(); fig.tight_layout()
FIGURE_DIR = REPORTS_DIR / E2_RUN.name
FIGURE_DIR.mkdir(exist_ok=True)
fig.savefig(FIGURE_DIR / "e2-bce-curve.png", dpi=180)
plt.show()
print("Best epoch:", int(best.epoch), "— chosen by validation BCE, not test")


## 3. Đóng băng rồi so sánh trên CASIA test đã mở
Bảng này không phải xác nhận độc lập mới. Không chọn thêm biến thể bằng kết quả test.


In [ ]:
#@title 3.1 Khóa artifact rồi score E2 một lần; tái dùng E1 đã có
freeze = {
    "config_sha256": manifest_checksum(E2_RUN / "config.yaml"),
    "checkpoint_sha256": manifest_checksum(E2_RUN / "best.ckpt"),
    "threshold_sha256": manifest_checksum(E2_RUN / "threshold.json"),
    "manifest_sha256": manifest_checksum(DEPTH_MANIFEST),
    "evaluation_status": "CASIA development test already observed in round 3",
}
freeze_path = E2_RUN / "comparison_freeze.json"
if freeze_path.is_file():
    assert json.loads(freeze_path.read_text()) == freeze, "Frozen E2 artifact changed"
else:
    freeze_path.write_text(json.dumps(freeze, indent=2), encoding="utf-8")
assert (E1_RUN / "test_metrics.json").is_file() and (E1_RUN / "test_scores.csv").is_file()
if not (E2_RUN / "test_metrics.json").is_file():
    run([sys.executable, "scripts/score_checkpoint.py", "--run-dir", E2_RUN, "--split", "test"], cwd=REPO_ROOT)
comparison = pd.DataFrame([
    {"model": "E1 fixed mean", **json.loads((E1_RUN / "test_metrics.json").read_text())},
    {"model": "E2 frozen head", **json.loads((E2_RUN / "test_metrics.json").read_text())},
]).set_index("model")
comparison["delta_acer_vs_E1"] = comparison.acer - comparison.loc["E1 fixed mean", "acer"]
comparison.to_csv(FIGURE_DIR / "e1-e2-development-test.csv")
display(comparison.style.format({"apcer":"{:.4%}", "bpcer":"{:.4%}", "acer":"{:.4%}", "eer":"{:.4%}", "auc":"{:.6f}", "threshold":"{:.8f}", "delta_acer_vs_E1":"{:+.4%}"}))
print("One seed, CASIA development protocol, test previously observed; not SOTA evidence.")


In [ ]:
#@title 3.2 E2 sửa được gì và làm sai thêm gì? So sánh theo cùng video
def paired_errors(split):
    first = pd.read_csv(E1_RUN / f"{split}_scores.csv", dtype={"video_id": str}).rename(columns={"score":"score_e1"})
    second = pd.read_csv(E2_RUN / f"{split}_scores.csv", dtype={"video_id": str}).rename(columns={"score":"score_e2"})
    result = first.merge(second, on=["video_id", "label"], validate="one_to_one")
    assert len(result) == len(first) == len(second)
    result["pred_e1"] = (result.score_e1 >= e1_threshold["threshold"]).astype(int)
    result["pred_e2"] = (result.score_e2 >= e2_threshold["threshold"]).astype(int)
    result["correct_e1"] = result.pred_e1 == result.label
    result["correct_e2"] = result.pred_e2 == result.label
    result["change"] = np.select([
        ~result.correct_e1 & result.correct_e2,
        result.correct_e1 & ~result.correct_e2,
        ~result.correct_e1 & ~result.correct_e2,
    ], ["fixed_by_E2", "new_error_E2", "both_wrong"], default="both_correct")
    metadata = pd.read_csv(MANIFEST, dtype={"video_id": str})
    columns = ["video_id", "label", "attack_type", "quality"]
    metadata = metadata.loc[metadata.split == split, columns].drop_duplicates()
    return result.merge(metadata, on=["video_id", "label"], validate="one_to_one")

errors = paired_errors("test")
errors.to_csv(FIGURE_DIR / "e1-e2-paired-errors.csv", index=False)
display(errors.groupby(["label", "attack_type", "change"]).size().rename("videos").to_frame())
display(errors[errors.change != "both_correct"])
val_errors = paired_errors("val")
val_errors.to_csv(FIGURE_DIR / "e1-e2-validation-paired-errors.csv", index=False)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, column, title, threshold in zip(axes, ["score_e1", "score_e2"], ["E1 mean depth", "E2 head sigmoid"], [e1_threshold["threshold"], e2_threshold["threshold"]]):
    for label, name in [(0, "attack"), (1, "live")]:
        ax.hist(errors.loc[errors.label == label, column], bins=25, alpha=.55, label=name)
    ax.axvline(threshold, color="black", linestyle="--", label="validation threshold")
    ax.set(xlabel="Score (different scales)", ylabel="Videos", title=title)
    ax.legend()
fig.tight_layout(); fig.savefig(FIGURE_DIR / "e1-e2-score-distributions.png", dpi=180); plt.show()


In [ ]:
#@title 3.3 Params và latency — RGB→score, batch 1, cùng GPU
e1 = build_model(e1_config).to(device).eval()
e1.load_state_dict(e1_state["model"])
e2 = build_model(config).to(device).eval()
trained = torch.load(E2_RUN / "best.ckpt", map_location=device, weights_only=False)
e2.load_state_dict(trained["model"])
sample = torch.randn(1, 3, 256, 256, device=device)

@torch.no_grad()
def latency(model, tensor, warmup=10, repeats=50):
    for _ in range(warmup):
        model(tensor)
    torch.cuda.synchronize()
    measurements = []
    for _ in range(repeats):
        torch.cuda.synchronize()
        start = time.perf_counter()
        model(tensor)
        torch.cuda.synchronize()
        measurements.append((time.perf_counter() - start) * 1000)
    return {"latency_p50_ms": float(np.median(measurements)), "latency_p95_ms": float(np.percentile(measurements, 95))}

efficiency = pd.DataFrame([
    {"model":"E1 fixed mean", "parameters":sum(p.numel() for p in e1.parameters()), **latency(e1, sample)},
    {"model":"E2 frozen head", "parameters":sum(p.numel() for p in e2.parameters()), **latency(e2, sample)},
]).set_index("model")
efficiency["fps_from_p50"] = 1000 / efficiency.latency_p50_ms
efficiency.to_csv(FIGURE_DIR / "e1-e2-efficiency.csv")
display(efficiency)
print("Head parameters:", sum(p.numel() for p in e2.head.parameters()))
print("Added params (%):", 100 * (efficiency.parameters.iloc[1] / efficiency.parameters.iloc[0] - 1))
print("Latency includes model forward only; no crop, disk load, CPU-to-GPU transfer or webcam.")
print("GPU:", torch.cuda.get_device_name(0), "torch:", torch.__version__)


## 4. Chuẩn bị báo cáo

Lấy bảng test phát triển, BCE curve, paired error CSV và efficiency CSV trong
`/content/drive/MyDrive/face-pad/reports/<E2_RUN_ID>`.

- Nêu rõ E1 và E2 có cùng backbone/checkpoint/depth map; thay cách đọc map.
- ACER/APCER/BPCER và threshold đánh giá tại operating point; AUC/EER xét khả năng phân tách.
- Báo cả video được sửa và lỗi mới; improvement có thể âm hoặc bằng 0.
- Một seed và CASIA test đã mở chỉ cho kết luận phát triển. UCDCN là tiền lệ;
  chưa so với mô hình hiện đại, nhiều seed hay benchmark độc lập.
- Không so BCE của E2 với depth loss của E1 như thể hai loss có cùng ý nghĩa.
- Nếu run chưa xong, báo trạng thái thật; không điền số giả.

Notebook sạch này chưa chứa output train. File E1 đã chạy vẫn được giữ riêng.
